# 01 — EDA датасета BraTS 2021 (FLAIR + T1ce) для NeuroSeg Core

Ноутбук проводит разведочный анализ **только** открытого датасета BraTS 2021 (Kaggle: `dschettler8845/brats-2021-task1`) под задачу 3D-сегментации глиом по **2 модальностям**.

**Что делает ноутбук**

| Блок | Содержание |
|---|---|
| 1 | Загрузка данных (kagglehub), распаковка **только** нужных файлов, индекс кейсов |
| 2 | Единый проход по всем кейсам (многопоточный): заголовки, целостность, NaN/Inf, метки, объёмы, bbox, интенсивности |
| 3 | Состав, объём и качество данных (таблицы + сводка проверок) |
| 4 | Метки и дисбаланс классов `WT` / `TC` / `ET`, геометрия опухоли, карта локализации |
| 5 | Интенсивности `FLAIR` / `T1ce`: здоровая ткань vs опухоль, разброс между кейсами |
| 6 | 2D/3D срезы с наложением контуров масок |
| 7 | Предварительный сплит на 5 фолдов (`StratifiedGroupKFold`) |
| 8 | Сводка чисел для дизайн-дока (`[ожидает прогона на GPU]`) и упаковка графиков в `docs/images/` |

**Правила анализа**

- Используются строго каналы `0` (`FLAIR`) и `2` (`T1ce`) из порядка `[FLAIR, T1, T1ce, T2]`. Файлы `T1` и `T2` не читаются.
- Метки BraTS `{0, 1 (NCR), 2 (ED), 4 (ET)}` приводятся к вложенным субрегионам: `WT = {1,2,4}`, `TC = {1,4}`, `ET = {4}`.
- Код архитектуры и обучения модели в ноутбук **не входит**.

## 0. Окружение и конфигурация

In [ ]:
!pip -q install nibabel kagglehub

import os, json, tarfile, zipfile, threading, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
from scipy import ndimage
from tqdm.auto import tqdm
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11, "axes.labelsize": 10})

print("nibabel", nib.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

In [ ]:
# ---------------- Конфигурация ----------------
# Порядок каналов в BraTS: 0=FLAIR, 1=T1, 2=T1ce, 3=T2. Для EDA берём строго каналы 0 и 2.
MODALITY_ORDER = ["flair", "t1", "t1ce", "t2"]
TARGET_CHANNELS = [0, 2]
TARGET_MODALITIES = [MODALITY_ORDER[c] for c in TARGET_CHANNELS]
IGNORED_MODALITIES = [
    m for i, m in enumerate(MODALITY_ORDER) if i not in TARGET_CHANNELS
]
assert TARGET_MODALITIES == ["flair", "t1ce"], TARGET_MODALITIES
PRETTY = {"flair": "FLAIR", "t1ce": "T1ce"}

# Метки BraTS 2021: 0=фон, 1=NCR (некротическое ядро), 2=ED (отёк), 4=ET (контрастируемая опухоль)
KNOWN_LABELS = (0, 1, 2, 4)
EXPECTED_SHAPE_STR = "240x240x155"
EXPECTED_SHAPE = (240, 240, 155)

SEED = 42
MAX_CASES = (
    None  # int -> быстрый прогон на подвыборке (например, 40); None -> весь датасет
)
N_WORKERS = 4  # число потоков сканирования (gzip отпускает GIL)
SAMPLES_PER_CLASS = (
    2000  # сколько вокселей на кейс/класс сохраняем для гистограмм интенсивности
)
ET_TINY_VOX = 100  # порог "крошечной" ET-области, вокселей
N_FOLDS = 5
FORCE_RESCAN = False  # True -> игнорировать кэш и заново пройти по данным

WORK_DIR = ...
IMG_DIR = WORK_DIR / "docs" / "images"
TABLE_DIR = WORK_DIR / "docs" / "eda_tables"
EXTRACT_DIR = WORK_DIR / "brats_extracted"
for d in (IMG_DIR, TABLE_DIR, EXTRACT_DIR):
    d.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)


# ---------------- Утилиты ----------------
def labels_to_regions(seg):
    """Метки BraTS -> вложенные бинарные субрегионы (WT, TC, ET)."""
    seg = np.asarray(seg)
    wt = np.isin(seg, (1, 2, 4))  # Whole Tumor = NCR + ED + ET
    tc = np.isin(seg, (1, 4))  # Tumor Core  = NCR + ET
    et = seg == 4  # Enhancing Tumor
    return wt, tc, et


def fmt_tuple(t):
    return "x".join(f"{v:g}" for v in t)


def save_fig(fig, name):
    path = IMG_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved -> {path}")
    plt.show()


def conclude(title, bullets):
    """Печатает автоматически сформированные (на основе реальных чисел) выводы к графику."""
    text = f"**Вывод — {title}**\n\n" + "\n".join(f"- {b}" for b in bullets)
    display(Markdown(text))


print(
    "Целевые каналы:",
    dict(zip(TARGET_CHANNELS, TARGET_MODALITIES)),
    "| игнорируются:",
    IGNORED_MODALITIES,
)

## 1. Загрузка данных и индексация кейсов

Загрузка через `kagglehub` (код из `discussion/EDA.ipynb`). Из архивов распаковываются **только** файлы `*_flair.nii*`, `*_t1ce.nii*`, `*_seg.nii*` (каналы `0` и `2` + разметка); `T1`/`T2` пропускаются, что экономит диск и время. Архивы валидационной выборки (без публичной разметки) пропускаются.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dschettler8845/brats-2021-task1")

print("Path to dataset files:", path)

In [ ]:
import re

DATA_ROOT = Path(path)
# Нужны только FLAIR (канал 0), T1ce (канал 2) и разметка seg. T1/T2 пропускаем.
WANTED_RE = re.compile(
    r"(?P<case>BraTS\d*_?\d+)_(?P<kind>flair|t1ce|seg)\.nii(\.gz)?$", re.IGNORECASE
)
ARCHIVE_EXT = (".tar", ".tar.gz", ".tgz", ".zip")


def list_files(root):
    out = []
    for r, _, files in os.walk(root):
        for f in files:
            out.append(Path(r) / f)
    return out


all_files = list_files(DATA_ROOT)
archives = [p for p in all_files if p.name.lower().endswith(ARCHIVE_EXT)]
print(f"Файлов в датасете: {len(all_files)}, из них архивов: {len(archives)}")
for p in sorted(archives):
    print(f"  {p.name:<45} {p.stat().st_size / 1e9:6.2f} GB")
print(
    "Прочие файлы (csv/json/txt):",
    [p.name for p in all_files if p.suffix.lower() in (".csv", ".json", ".txt")],
)


def extract_wanted_from_tar(archive_path):
    n = 0
    with tarfile.open(archive_path, "r:*") as tf:
        for m in tf:
            if not m.isfile():
                continue
            mt = WANTED_RE.search(Path(m.name).name)
            if not mt:
                continue
            dest_dir = EXTRACT_DIR / mt.group("case")
            dest_dir.mkdir(parents=True, exist_ok=True)
            dest = dest_dir / Path(m.name).name
            if dest.exists() and dest.stat().st_size == m.size:
                continue
            with tf.extractfile(m) as src, open(dest, "wb") as dst:
                while True:
                    chunk = src.read(1 << 24)
                    if not chunk:
                        break
                    dst.write(chunk)
            n += 1
    return n


def extract_wanted_from_zip(archive_path):
    n = 0
    with zipfile.ZipFile(archive_path) as zf:
        for info in zf.infolist():
            mt = WANTED_RE.search(Path(info.filename).name)
            if info.is_dir() or not mt:
                continue
            dest_dir = EXTRACT_DIR / mt.group("case")
            dest_dir.mkdir(parents=True, exist_ok=True)
            dest = dest_dir / Path(info.filename).name
            if dest.exists() and dest.stat().st_size == info.file_size:
                continue
            with zf.open(info) as src, open(dest, "wb") as dst:
                while True:
                    chunk = src.read(1 << 24)
                    if not chunk:
                        break
                    dst.write(chunk)
            n += 1
    return n


SKIPPED_ARCHIVES = []
for ap in sorted(archives):
    low = ap.name.lower()
    # Валидационная/тестовая выборка BraTS не содержит публичной разметки -> для EDA не используется
    if "valid" in low or "test" in low:
        SKIPPED_ARCHIVES.append(ap.name)
        continue
    print("Распаковка нужных файлов из", ap.name, "...")
    n = (
        extract_wanted_from_zip(ap)
        if low.endswith(".zip")
        else extract_wanted_from_tar(ap)
    )
    print(f"  извлечено файлов: {n}")
print("Пропущенные архивы (нет разметки):", SKIPPED_ARCHIVES)

# ---- Индекс кейсов: ищем и в исходной папке (если данные уже распакованы), и в EXTRACT_DIR ----
index = {}
for p in list_files(DATA_ROOT) + list_files(EXTRACT_DIR):
    mt = WANTED_RE.search(p.name)
    if not mt:
        continue
    index.setdefault(mt.group("case"), {}).setdefault(mt.group("kind").lower(), p)

case_ids_all = sorted(index)
complete = [
    c for c in case_ids_all if all(k in index[c] for k in ("flair", "t1ce", "seg"))
]
incomplete = [c for c in case_ids_all if c not in complete]
print(
    f"\nНайдено кейсов: {len(case_ids_all)} | полных (FLAIR+T1ce+seg): {len(complete)} | неполных: {len(incomplete)}"
)
if incomplete:
    print("Неполные кейсы (первые 10):", incomplete[:10])

assert len(complete) > 0, (
    "Не найдено ни одного полного кейса. Проверьте структуру датасета (см. вывод выше)."
)

case_ids = complete if MAX_CASES is None else complete[:MAX_CASES]
print(f"В анализ идёт кейсов: {len(case_ids)} (MAX_CASES={MAX_CASES})")

## 2. Единый проход по данным (сканирование кейсов)

Для каждого кейса читаются только `FLAIR`, `T1ce` и `seg`. За один проход собираются:

- **заголовки**: `shape`, `spacing` (zooms), `affine`, ориентация осей, `dtype`;
- **целостность**: ошибки чтения/распаковки, `NaN`/`Inf`, отрицательные значения, пустой снимок;
- **метки**: набор уникальных значений `seg` и число вокселей каждой; приведение к `WT`/`TC`/`ET`;
- **геометрия**: bbox мозга и опухоли, число связных компонент, центроид опухоли;
- **интенсивности**: статистики по мозгу, здоровой ткани и подобластям опухоли, плюс случайные выборки вокселей для гистограмм.

Результат кешируется в `docs/eda_tables/`, повторный запуск ячейки не перечитывает данные (`FORCE_RESCAN = True` отключает кеш).

In [ ]:
import pickle

CACHE_PATH = TABLE_DIR / "scan_cache.pkl"
SAMPLE_CLASSES = (
    "healthy",
    "ed",
    "ncr",
    "et",
)  # классы, для которых сохраняем выборки вокселей


def _bbox(mask):
    """Возвращает (lo, hi) по каждой оси для ненулевой области либо None."""
    if not mask.any():
        return None
    lo, hi = [], []
    for ax in range(3):
        other = tuple(a for a in range(3) if a != ax)
        idx = np.flatnonzero(mask.any(axis=other))
        lo.append(int(idx[0]))
        hi.append(int(idx[-1]))
    return np.array(lo), np.array(hi)


def _stats(x):
    if x.size == 0:
        return dict(mean=np.nan, std=np.nan, p50=np.nan)
    return dict(mean=float(x.mean()), std=float(x.std()), p50=float(np.median(x)))


def _scan_headers(rec, imgs):
    for k, im in imgs.items():
        rec[f"shape_{k}"] = tuple(int(s) for s in im.shape)
        rec[f"zooms_{k}"] = tuple(round(float(z), 4) for z in im.header.get_zooms()[:3])
        rec[f"dtype_{k}"] = str(im.get_data_dtype())
    rec["orientation"] = "".join(nib.aff2axcodes(imgs["flair"].affine))
    rec["shape_match"] = rec["shape_flair"] == rec["shape_t1ce"] == rec["shape_seg"]
    rec["shape_expected"] = rec["shape_flair"] == EXPECTED_SHAPE
    rec["zooms_iso1"] = all(
        np.allclose(rec[f"zooms_{k}"], (1, 1, 1), atol=1e-3) for k in imgs
    )
    rec["zooms_match"] = rec["zooms_flair"] == rec["zooms_t1ce"] == rec["zooms_seg"]
    rec["affine_match"] = bool(
        np.allclose(imgs["flair"].affine, imgs["t1ce"].affine, atol=1e-3)
        and np.allclose(imgs["flair"].affine, imgs["seg"].affine, atol=1e-3)
    )


def _scan_integrity(rec, data, seg_raw):
    for k in ("flair", "t1ce"):
        x = data[k]
        rec[f"nan_{k}"] = int(np.isnan(x).sum())
        rec[f"inf_{k}"] = int(np.isinf(x).sum())
        rec[f"neg_{k}"] = int((x < 0).sum())
        if rec[f"nan_{k}"] or rec[f"inf_{k}"]:
            data[k] = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        rec[f"min_{k}"], rec[f"max_{k}"] = float(data[k].min()), float(data[k].max())
    rec["nan_seg"] = int(np.isnan(seg_raw).sum()) if seg_raw.dtype.kind == "f" else 0
    return np.rint(np.nan_to_num(seg_raw.astype(np.float32))).astype(np.int16)


def _scan_labels(rec, seg):
    lab, cnt = np.unique(seg, return_counts=True)
    rec["labels_present"] = ",".join(str(int(v)) for v in lab)
    rec["unknown_label_vox"] = int(
        sum(c for v, c in zip(lab, cnt) if int(v) not in KNOWN_LABELS)
    )
    wt, tc, et = labels_to_regions(seg)
    for name, m in (
        ("ncr", seg == 1),
        ("ed", seg == 2),
        ("et_lbl4", seg == 4),
        ("wt", wt),
        ("tc", tc),
        ("et", et),
    ):
        rec[f"vox_{name}"] = int(m.sum())
    return wt, tc, et


def _scan_brain_geometry(rec, nz, wt, et):
    brain = nz["flair"] | nz["t1ce"]
    rec["brain_vox"] = int(brain.sum())
    rec["brain_frac_of_volume"] = rec["brain_vox"] / brain.size
    rec["seg_outside_brain_vox"] = int((wt & ~brain).sum())
    bb = _bbox(brain)
    if bb is not None:
        ext = bb[1] - bb[0] + 1
        rec["brain_bbox_ext"] = tuple(int(e) for e in ext)
        rec["brain_bbox_frac"] = float(np.prod(ext) / brain.size)
    else:
        rec["brain_bbox_ext"], rec["brain_bbox_frac"] = (0, 0, 0), 0.0
    denom = max(rec["brain_vox"], 1)
    rec["wt_frac_of_brain"] = rec["vox_wt"] / denom
    rec["tc_frac_of_brain"] = rec["vox_tc"] / denom
    rec["et_frac_of_brain"] = rec["vox_et"] / denom

    proj = {}
    if rec["vox_wt"] > 0:
        wb = _bbox(wt)
        ext = wb[1] - wb[0] + 1
        rec["wt_bbox_ext"] = tuple(int(e) for e in ext)
        rec["wt_bbox_max"] = int(ext.max())
        cen = ndimage.center_of_mass(wt)
        rec["wt_centroid_norm"] = tuple(
            round(float(c) / s, 4) for c, s in zip(cen, wt.shape)
        )
        lbl, ncomp = ndimage.label(wt)
        rec["wt_ncomp"] = int(ncomp)
        sizes = np.bincount(lbl.ravel())[1:]
        rec["wt_largest_comp_frac"] = float(sizes.max() / sizes.sum())
        rec["et_ncomp"] = int(ndimage.label(et)[1]) if rec["vox_et"] > 0 else 0
        proj = {
            ax: wt.sum(axis=ax).astype(np.int32) for ax in range(3)
        }  # для карты локализации
    else:
        rec.update(
            wt_bbox_ext=(0, 0, 0),
            wt_bbox_max=0,
            wt_centroid_norm=(np.nan,) * 3,
            wt_ncomp=0,
            wt_largest_comp_frac=np.nan,
            et_ncomp=0,
        )
    return brain, proj


def _scan_intensity(rec, data, nz, brain, seg, wt, seed):
    r = np.random.default_rng(seed)
    samples = {}
    masks = {"healthy": brain & ~wt, "ed": seg == 2, "ncr": seg == 1, "et": seg == 4}
    for k in ("flair", "t1ce"):
        x, m_nz = data[k], nz[k]
        xb = x[m_nz]
        if xb.size:
            p1, p50, p99 = np.percentile(xb, [1, 50, 99])
        else:
            p1 = p50 = p99 = np.nan
        rec[f"{k}_brain_mean"] = float(xb.mean()) if xb.size else np.nan
        rec[f"{k}_brain_std"] = float(xb.std()) if xb.size else np.nan
        rec[f"{k}_p1"], rec[f"{k}_p50"], rec[f"{k}_p99"] = (
            float(p1),
            float(p50),
            float(p99),
        )
        for cls, m in masks.items():
            st = _stats(x[m & m_nz])
            rec[f"{k}_{cls}_mean"], rec[f"{k}_{cls}_std"], rec[f"{k}_{cls}_p50"] = (
                st["mean"],
                st["std"],
                st["p50"],
            )
        st = _stats(x[wt & m_nz])
        rec[f"{k}_wt_mean"], rec[f"{k}_wt_std"] = st["mean"], st["std"]
        for cls, m in masks.items():
            idx = np.flatnonzero((m & m_nz).ravel())
            if idx.size:
                take = (
                    idx
                    if idx.size <= SAMPLES_PER_CLASS
                    else r.choice(idx, SAMPLES_PER_CLASS, replace=False)
                )
                samples[(k, cls)] = x.ravel()[take].astype(np.float32)
    return samples


def scan_case(case_id, seed):
    rec = {"case_id": case_id, "status": "ok", "error": ""}
    files = index[case_id]
    rec["size_mb"] = (
        sum(os.path.getsize(files[k]) for k in ("flair", "t1ce", "seg")) / 1e6
    )
    samples, proj = {}, {}
    try:
        imgs = {k: nib.load(str(files[k])) for k in ("flair", "t1ce", "seg")}
        _scan_headers(rec, imgs)
        data = {
            k: np.asanyarray(imgs[k].dataobj).astype(np.float32)
            for k in ("flair", "t1ce")
        }
        seg_raw = np.asanyarray(imgs["seg"].dataobj)
        seg = _scan_integrity(rec, data, seg_raw)
        if not rec["shape_match"]:
            rec["status"] = "shape_mismatch"
            return rec, samples, proj
        wt, tc, et = _scan_labels(rec, seg)
        nz = {k: data[k] != 0 for k in data}
        brain, proj = _scan_brain_geometry(rec, nz, wt, et)
        samples = _scan_intensity(rec, data, nz, brain, seg, wt, seed)
    except Exception as e:  # битый gzip, повреждённый заголовок и т.п.
        rec["status"] = "error"
        rec["error"] = f"{type(e).__name__}: {e}"[:300]
    return rec, samples, proj


def run_scan():
    records = []
    samples_all = {(k, c): [] for k in TARGET_MODALITIES for c in SAMPLE_CLASSES}
    samples_cases = {(k, c): [] for k in TARGET_MODALITIES for c in SAMPLE_CLASSES}
    proj_sum = {
        0: np.zeros((240, 155), np.int64),
        1: np.zeros((240, 155), np.int64),
        2: np.zeros((240, 240), np.int64),
    }
    n_proj = 0
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = {
            ex.submit(scan_case, cid, SEED + i): cid for i, cid in enumerate(case_ids)
        }
        for fut in tqdm(as_completed(futs), total=len(futs), desc="scan"):
            rec, smp, proj = fut.result()
            records.append(rec)
            for key, arr in smp.items():
                samples_all[key].append(arr)
                samples_cases[key].append(
                    np.full(arr.shape, rec["case_id"], dtype=object)
                )
            if proj and all(proj[a].shape == proj_sum[a].shape for a in proj):
                for a in proj:
                    proj_sum[a] += proj[a]
                n_proj += 1
    samples = {
        k: (np.concatenate(v) if v else np.array([], np.float32))
        for k, v in samples_all.items()
    }
    samples_case = {
        k: (np.concatenate(v) if v else np.array([], dtype=object))
        for k, v in samples_cases.items()
    }
    df = pd.DataFrame(records).sort_values("case_id").reset_index(drop=True)
    return df, samples, samples_case, proj_sum, n_proj


cache = None
if CACHE_PATH.exists() and not FORCE_RESCAN:
    with open(CACHE_PATH, "rb") as f:
        cache = pickle.load(f)
    if cache.get("case_ids") == case_ids:
        df, samples, samples_case, proj_sum, n_proj = (
            cache[k] for k in ("df", "samples", "samples_case", "proj_sum", "n_proj")
        )
        print(f"Загружен кеш: {CACHE_PATH} ({len(df)} кейсов)")
    else:
        cache = None

if cache is None:
    df, samples, samples_case, proj_sum, n_proj = run_scan()
    with open(CACHE_PATH, "wb") as f:
        pickle.dump(
            dict(
                case_ids=case_ids,
                df=df,
                samples=samples,
                samples_case=samples_case,
                proj_sum=proj_sum,
                n_proj=n_proj,
            ),
            f,
        )
    df.to_csv(TABLE_DIR / "case_table.csv", index=False)
    print(f"Сохранено: {CACHE_PATH}")

print("Статусы кейсов:\n", df["status"].value_counts().to_string())

## 3. Состав, объём и качество данных

Проверяем контракт, который наш сервис ожидает от внешнего препроцессинга: одинаковые `shape`/`spacing`/`affine` у `FLAIR` и `T1ce`, изотропный воксель 1×1×1 мм, отсутствие `NaN`/`Inf`, читаемые файлы, корректный набор меток.

In [ ]:
ok = df[df["status"] == "ok"].copy()
n_total, n_ok = len(df), len(ok)

# ---------- Состав и объём ----------
total_gb = df["size_mb"].sum() / 1e3
print(f"Кейсов (пациентов) в анализе: {n_total}")
print(
    f"Объём FLAIR+T1ce+seg на диске: {total_gb:.1f} GB (в среднем {df['size_mb'].mean():.1f} MB на кейс)"
)
print(f"Кейсов с ошибкой чтения / несовпадением shape: {(df['status'] != 'ok').sum()}")
if (df["status"] != "ok").any():
    display(df.loc[df["status"] != "ok", ["case_id", "status", "error"]].head(20))

comp_tbl = pd.DataFrame(
    {
        "shape FLAIR": ok["shape_flair"].map(fmt_tuple).value_counts(),
        "spacing FLAIR, мм": ok["zooms_flair"].map(fmt_tuple).value_counts(),
        "spacing T1ce, мм": ok["zooms_t1ce"].map(fmt_tuple).value_counts(),
        "ориентация (axcodes)": ok["orientation"].value_counts(),
        "dtype FLAIR": ok["dtype_flair"].value_counts(),
        "dtype seg": ok["dtype_seg"].value_counts(),
    }
)
for col in comp_tbl.columns:
    print(f"\n--- {col} ---")
    print(comp_tbl[col].dropna().astype(int).to_string())


# ---------- Контракт данных (аналог Input Guardrails) ----------
def n_bad(mask):
    return int(np.asarray(mask, dtype=bool).sum())


# Проверки заголовков делаем по всем кейсам, у которых заголовки удалось прочитать
# (включая кейсы со статусом shape_mismatch, которые исключены из `ok`).
hdr = df[df["status"] != "error"].copy()
for c in ("shape_match", "shape_expected", "zooms_iso1", "zooms_match", "affine_match"):
    hdr[c] = hdr[c].astype(bool)

checks = {
    "Все 3 файла (FLAIR, T1ce, seg) читаются": n_bad(df["status"] == "error"),
    "shape FLAIR == T1ce == seg": n_bad(~hdr["shape_match"]),
    f"shape == {EXPECTED_SHAPE_STR}": n_bad(~hdr["shape_expected"]),
    "spacing == 1x1x1 мм (все файлы)": n_bad(~hdr["zooms_iso1"]),
    "spacing совпадает между FLAIR/T1ce/seg": n_bad(~hdr["zooms_match"]),
    "affine совпадает между FLAIR/T1ce/seg": n_bad(~hdr["affine_match"]),
    "NaN в FLAIR": n_bad(ok["nan_flair"] > 0),
    "NaN в T1ce": n_bad(ok["nan_t1ce"] > 0),
    "Inf в FLAIR/T1ce": n_bad((ok["inf_flair"] > 0) | (ok["inf_t1ce"] > 0)),
    "Отрицательные интенсивности (FLAIR/T1ce)": n_bad(
        (ok["neg_flair"] > 0) | (ok["neg_t1ce"] > 0)
    ),
    "Метки вне {0,1,2,4}": n_bad(ok["unknown_label_vox"] > 0),
    "Пустая разметка (опухоль не найдена)": n_bad(ok["vox_wt"] == 0),
    "Вокселей WT вне ненулевой области мозга": n_bad(ok["seg_outside_brain_vox"] > 0),
    "Пустой снимок (весь мозг = 0)": n_bad(ok["brain_vox"] == 0),
}
qc = pd.DataFrame({"кейсов с нарушением": checks})
qc["доля, %"] = (qc["кейсов с нарушением"] / max(n_total, 1) * 100).round(2)
qc["статус"] = np.where(qc["кейсов с нарушением"] == 0, "OK", "ВНИМАНИЕ")
display(qc)
qc.to_csv(TABLE_DIR / "data_quality_checks.csv")

# ---------- Дубликаты (грубая проверка по сигнатуре кейса) ----------
sig_cols = [
    "brain_vox",
    "vox_wt",
    "vox_tc",
    "vox_et",
    "flair_brain_mean",
    "t1ce_brain_mean",
]
n_dup = int(ok.duplicated(subset=sig_cols, keep=False).sum())
print(f"\nКейсов с идентичной сигнатурой (возможные дубликаты): {n_dup}")

# ---------- Визуализация: размер мозга и распределение размеров файлов ----------
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(ok["brain_frac_of_volume"] * 100, bins=40, ax=axes[0], color="steelblue")
axes[0].set_xlabel("Мозг, % вокселей тома 240x240x155")
axes[0].set_title("Доля вокселей мозга в объёме")
sns.histplot(
    ok["brain_bbox_ext"].map(lambda t: t[0]),
    bins=30,
    ax=axes[1],
    color="indianred",
    label="X",
)
sns.histplot(
    ok["brain_bbox_ext"].map(lambda t: t[1]),
    bins=30,
    ax=axes[1],
    color="seagreen",
    label="Y",
)
sns.histplot(
    ok["brain_bbox_ext"].map(lambda t: t[2]),
    bins=30,
    ax=axes[1],
    color="goldenrod",
    label="Z",
)
axes[1].set_xlabel("Размер bbox мозга по оси, вокс.")
axes[1].set_title("Габариты мозга (bbox)")
axes[1].legend()
sns.histplot(ok["size_mb"], bins=30, ax=axes[2], color="slategray")
axes[2].set_xlabel("MB на кейс (FLAIR+T1ce+seg)")
axes[2].set_title("Размер файлов на кейс")
fig.tight_layout()
save_fig(fig, "eda_01_brain_extent.png")

bf = ok["brain_frac_of_volume"] * 100
bb_frac = ok["brain_bbox_frac"] * 100
conclude(
    "состав и качество данных",
    [
        f"В анализ вошло **{n_total}** кейсов, из них успешно прочитано **{n_ok}**; суммарно {total_gb:.1f} GB для 2 модальностей и разметки.",
        f"Мозг занимает в среднем **{bf.mean():.1f}%** вокселей тома (min {bf.min():.1f}%, max {bf.max():.1f}%), bbox мозга — в среднем {bb_frac.mean():.1f}% тома. "
        "Значит, `CropForegroundd` заметно сократит число обрабатываемых вокселей.",
        f"Проверки контракта: нарушений по shape/spacing/affine — {checks['shape FLAIR == T1ce == seg'] + checks['spacing == 1x1x1 мм (все файлы)'] + checks['affine совпадает между FLAIR/T1ce/seg']}, "
        f"по NaN/Inf — {checks['NaN в FLAIR'] + checks['NaN в T1ce'] + checks['Inf в FLAIR/T1ce']}, по читаемости файлов — {checks['Все 3 файла (FLAIR, T1ce, seg) читаются']}. "
        "Если все значения равны 0, то BraTS уже находится в формате, который ожидает наш модуль, и входные guardrails нужны скорее для продакшн-данных, чем для этого датасета.",
    ],
)

## 4. Метки, субрегионы `WT` / `TC` / `ET` и дисбаланс классов

Исходные метки BraTS (`1=NCR`, `2=ED`, `4=ET`) приведены к вложенным субрегионам `WT ⊃ TC ⊃ ET`. Ниже оцениваем:

1. долю вокселей каждого класса в объёме и в мозге (дисбаланс «фон / опухоль»);
2. абсолютные объёмы субрегионов (в мл, т.к. воксель 1 мм³ → 1000 вокселей = 1 мл);
3. число пациентов без `ET` (и с «крошечной» `ET`);
4. геометрию (число связных компонент, размер bbox) и локализацию опухоли.

In [ ]:
ok = df[df["status"] == "ok"].copy()
VOX_ML = 1000.0  # 1 воксель = 1 мм^3 -> 1000 вокселей = 1 мл

# ---------- Сводка по долям классов ----------
total_vox = ok["shape_flair"].map(lambda s: int(np.prod(s))).astype(np.int64)
frac_tbl = pd.DataFrame(
    {
        "WT, % объёма": ok["vox_wt"] / total_vox * 100,
        "TC, % объёма": ok["vox_tc"] / total_vox * 100,
        "ET, % объёма": ok["vox_et"] / total_vox * 100,
        "WT, % мозга": ok["wt_frac_of_brain"] * 100,
        "TC, % мозга": ok["tc_frac_of_brain"] * 100,
        "ET, % мозга": ok["et_frac_of_brain"] * 100,
    }
)
display(frac_tbl.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(3))

# Глобальный дисбаланс (на уровне вокселей по всем кейсам)
brain_sum = ok["brain_vox"].sum()
global_frac = pd.Series(
    {
        "Фон/здоровая ткань мозга": (brain_sum - ok["vox_wt"].sum()) / brain_sum * 100,
        "WT": ok["vox_wt"].sum() / brain_sum * 100,
        "TC": ok["vox_tc"].sum() / brain_sum * 100,
        "ET": ok["vox_et"].sum() / brain_sum * 100,
        "NCR (метка 1)": ok["vox_ncr"].sum() / brain_sum * 100,
        "ED (метка 2)": ok["vox_ed"].sum() / brain_sum * 100,
    },
    name="% вокселей мозга (по всем кейсам)",
)
display(global_frac.round(3).to_frame())

# ---------- Объёмы субрегионов, мл ----------
vol = pd.DataFrame(
    {
        "WT": ok["vox_wt"] / VOX_ML,
        "TC": ok["vox_tc"] / VOX_ML,
        "ET": ok["vox_et"] / VOX_ML,
        "NCR": ok["vox_ncr"] / VOX_ML,
        "ED": ok["vox_ed"] / VOX_ML,
    }
)
display(vol.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2))

# ---------- Пациенты без ET и с крошечной ET ----------
ok["has_et"] = ok["vox_et"] > 0
ok["has_tc"] = ok["vox_tc"] > 0
n_no_et = int((~ok["has_et"]).sum())
n_tiny_et = int(((ok["vox_et"] > 0) & (ok["vox_et"] < ET_TINY_VOX)).sum())
n_no_tc = int((~ok["has_tc"]).sum())
print(f"Без ET (vox_et == 0): {n_no_et} из {n_ok} ({n_no_et / n_ok * 100:.1f}%)")
print(f"ET < {ET_TINY_VOX} вокс.: {n_tiny_et} ({n_tiny_et / n_ok * 100:.1f}%)")
print(f"Без TC (vox_tc == 0): {n_no_tc}")
print("Набор меток в кейсах:\n", ok["labels_present"].value_counts().to_string())

# Графики 1: распределение объёмов
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
colors = {"WT": "#3b82f6", "TC": "#f59e0b", "ET": "#ef4444"}
for ax, reg in zip(axes[0], ["WT", "TC", "ET"]):
    sns.histplot(vol[reg], bins=40, ax=ax, color=colors[reg])
    ax.axvline(
        vol[reg].median(),
        color="k",
        ls="--",
        lw=1,
        label=f"медиана {vol[reg].median():.1f} мл",
    )
    ax.set_title(f"Объём {reg}")
    ax.set_xlabel("мл")
    ax.legend()

# log-шкала объёмов (ящик с усами)
long = vol[["WT", "TC", "ET"]].melt(var_name="регион", value_name="мл")
long["мл"] = long["мл"].clip(lower=1e-3)
sns.boxplot(
    data=long,
    x="регион",
    y="мл",
    hue="регион",
    palette=colors,
    ax=axes[1, 0],
    legend=False,
)
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("Объёмы субрегионов (log-шкала; 0 → 1e-3)")

# доля в мозге
pct = frac_tbl[["WT, % мозга", "TC, % мозга", "ET, % мозга"]].copy()
pct.columns = ["WT", "TC", "ET"]
sns.boxplot(
    data=pct.melt(var_name="регион", value_name="%"),
    x="регион",
    y="%",
    hue="регион",
    palette=colors,
    ax=axes[1, 1],
    legend=False,
)
axes[1, 1].set_title("Доля региона от вокселей мозга, %")

# Столбики: доля классов (глобально, log)
bars = global_frac[["WT", "TC", "ET"]]
axes[1, 2].bar(bars.index, bars.values, color=[colors[i] for i in bars.index])
axes[1, 2].set_yscale("log")
axes[1, 2].set_ylabel("% вокселей мозга (log)")
axes[1, 2].set_title("Дисбаланс: опухоль vs весь мозг (глобально)")
for i, v in enumerate(bars.values):
    axes[1, 2].text(i, v, f"{v:.2f}%", ha="center", va="bottom")
fig.tight_layout()
save_fig(fig, "eda_02_region_volumes.png")

wt_med, tc_med, et_med = vol["WT"].median(), vol["TC"].median(), vol["ET"].median()
conclude(
    "объёмы субрегионов и дисбаланс",
    [
        f"Медианные объёмы: `WT` {wt_med:.1f} мл, `TC` {tc_med:.1f} мл, `ET` {et_med:.1f} мл — порядок величин различается, `ET` — самый мелкий и самый «хрупкий» для метрик регион.",
        f"Глобально опухоль (`WT`) занимает **{global_frac['WT']:.2f}%** вокселей мозга, `TC` — {global_frac['TC']:.2f}%, `ET` — {global_frac['ET']:.2f}%: "
        "дисбаланс «фон / опухоль» выражен, а чистая воксельная кросс-энтропия будет склонна предсказывать фон → нужен составной лосс (`Dice` + `CE`).",
        f"Пациентов без `ET`: **{n_no_et}** ({n_no_et / n_ok * 100:.1f}%), с «крошечной» `ET` (< {ET_TINY_VOX} вокс.): {n_tiny_et}. "
        "Для таких кейсов Dice по `ET` не определён/нулевой — нужен явный протокол (пустое GT и пустой прогноз = Dice 1) и стратификация фолдов по наличию `ET`.",
        f"Разброс объёмов `WT` велик (5–95 перцентиль: {vol['WT'].quantile(0.05):.1f}–{vol['WT'].quantile(0.95):.1f} мл) → "
        "метрики будут сильно зависеть от размера опухоли, необходим разбор качества по бинам объёма.",
    ],
)

In [ ]:
# ---------- Геометрия: связные компоненты, bbox, ET/TC vs WT ----------
geo = ok[ok["vox_wt"] > 0].copy()
geo["et_share_of_wt"] = geo["vox_et"] / geo["vox_wt"]
geo["tc_share_of_wt"] = geo["vox_tc"] / geo["vox_wt"]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
sns.histplot(geo["wt_ncomp"].clip(upper=15), discrete=True, ax=axes[0], color="purple")
axes[0].set_xlabel("Число связных компонент WT (>=15 объединено)")
axes[0].set_title("Фрагментация опухоли")
sns.histplot(geo["wt_bbox_max"], bins=30, ax=axes[1], color="teal")
axes[1].set_xlabel("Макс. размер bbox WT, вокс. (=мм)")
axes[1].set_title("Размер опухоли (bbox)")
sns.histplot(geo["wt_largest_comp_frac"], bins=30, ax=axes[2], color="darkorange")
axes[2].set_xlabel("Доля крупнейшей компоненты в WT")
axes[2].set_title("Доминирует ли одна компонента")
sns.scatterplot(
    data=geo,
    x=geo["vox_wt"] / VOX_ML,
    y=geo["vox_et"] / VOX_ML,
    alpha=0.5,
    s=14,
    ax=axes[3],
    color="crimson",
)
axes[3].set_xscale("log")
axes[3].set_yscale("symlog", linthresh=0.1)
axes[3].set_xlabel("Объём WT, мл (log)")
axes[3].set_ylabel("Объём ET, мл (symlog)")
axes[3].set_title("ET vs WT")
fig.tight_layout()
save_fig(fig, "eda_03_geometry.png")

multi = float((geo["wt_ncomp"] > 1).mean() * 100)
bbmax = geo["wt_bbox_max"]
conclude(
    "геометрия опухоли",
    [
        f"У **{multi:.1f}%** кейсов `WT` состоит из более чем одной связной компоненты (медиана компонент — {geo['wt_ncomp'].median():.0f}). "
        "Следовательно, постобработка `KeepLargestConnectedComponent` может удалить настоящие очаги: порог надо подбирать по объёму, а не оставлять только максимальную компоненту.",
        f"Размер bbox опухоли: медиана {bbmax.median():.0f} мм, 95-й перцентиль {bbmax.quantile(0.95):.0f} мм, максимум {bbmax.max():.0f} мм. "
        "Это ориентир для выбора `roi_size` патча (128³ покрывает типичную опухоль целиком, но крупные случаи могут не поместиться).",
        f"Доля `ET` в `WT`: медиана {geo['et_share_of_wt'].median() * 100:.1f}%, `TC` в `WT`: медиана {geo['tc_share_of_wt'].median() * 100:.1f}%. "
        "Ручной контроль: график ET vs WT показывает, что у части кейсов `ET` = 0 при ненулевой `WT` (нижняя полоса).",
    ],
)

# ---------- Карта локализации опухолей ----------
titles = {
    0: "Сагиттальная проекция (суммарно по всем WT)",
    1: "Корональная проекция",
    2: "Аксиальная проекция",
}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax_i, ax in zip(range(3), axes):
    heat = proj_sum[ax_i].astype(float) / max(n_proj, 1)
    im = ax.imshow(np.rot90(heat), cmap="inferno")
    ax.set_title(titles[ax_i])
    ax.axis("off")
    fig.colorbar(
        im, ax=ax, fraction=0.046, label="ср. число вокс. WT на пикс. проекции"
    )
fig.tight_layout()
save_fig(fig, "eda_04_tumor_location_heatmap.png")

cen = np.array(geo["wt_centroid_norm"].tolist())
conclude(
    "локализация опухоли",
    [
        f"Средние нормированные координаты центроидов `WT` по осям (x, y, z): {np.nanmean(cen, 0).round(2).tolist()}; стандартное отклонение — {np.nanstd(cen, 0).round(2).tolist()}.",
        "Тепловая карта показывает, где в стандартном пространстве BraTS чаще встречается опухоль. Если плотность неоднородна (смещение к лобным/височным долям), то случайное кропирование патчей надо смещать в сторону опухоли (`RandCropByPosNegLabeld` с `pos > neg`), иначе большая часть патчей будет содержать только фон.",
    ],
)

## 5. Интенсивности `FLAIR` и `T1ce`: здоровая ткань vs опухоль

BraTS содержит «сырые» интенсивности без физической шкалы (в отличие от КТ с единицами Хаунсфилда), поэтому абсолютные значения разных сканеров несопоставимы. Анализируем:

1. гистограммы вокселей в здоровой ткани мозга и в подобластях опухоли (`ED`, `NCR`, `ET`) — насколько классы различимы по каждой модальности;
2. межкейсовый разброс статистик мозга (медиана, 99-й перцентиль) — масштаб доменного сдвига и обоснование нормализации `NormalizeIntensityd`;
3. контраст «опухоль / здоровая ткань» по кейсам (кто несёт информацию о каком субрегионе).

Для гистограмм использована случайная выборка вокселей (до `SAMPLES_PER_CLASS` на кейс/класс), чтобы все кейсы вносили сопоставимый вклад. Для сопоставимости формы распределения по шкалам для каждого кейса отдельно применена нормализация по 99-му перцентилю мозга (панель «после нормализации»).

In [ ]:
CLS_NAMES = {
    "healthy": "Здоровая ткань мозга",
    "ed": "ED (отёк)",
    "ncr": "NCR (некроз)",
    "et": "ET (контраст.)",
}
CLS_COLORS = {"healthy": "#6b7280", "ed": "#3b82f6", "ncr": "#a855f7", "et": "#ef4444"}

for k in TARGET_MODALITIES:
    for c in SAMPLE_CLASSES:
        print(
            f"{PRETTY[k]:6s} | {CLS_NAMES[c]:<22} | выборка вокселей: {len(samples[(k, c)]):>9,}"
        )

# ---------- Гистограммы сырых интенсивностей ----------
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for j, k in enumerate(TARGET_MODALITIES):
    hi = np.percentile(samples[(k, "healthy")], 99.5)
    bins = np.linspace(0, hi * 1.3, 120)
    for c in SAMPLE_CLASSES:
        x = samples[(k, c)]
        if x.size == 0:
            continue
        sns.histplot(
            x,
            bins=bins,
            stat="density",
            element="step",
            fill=(c != "healthy"),
            alpha=0.25,
            color=CLS_COLORS[c],
            label=CLS_NAMES[c],
            ax=axes[0, j],
        )
    axes[0, j].set_title(f"{PRETTY[k]}: сырые интенсивности (пул всех кейсов)")
    axes[0, j].set_xlabel("интенсивность")
    axes[0, j].legend()

    # после нормализации кейса на его p99 мозга: убираем межкейсовый масштаб
    scale = df.set_index("case_id")[f"{k}_p99"]
    for c in SAMPLE_CLASSES:
        x, cid = samples[(k, c)], samples_case[(k, c)]
        if x.size == 0:
            continue
        xn = x / scale.reindex(cid).values
        sns.histplot(
            xn,
            bins=np.linspace(0, 1.4, 120),
            stat="density",
            element="step",
            fill=(c != "healthy"),
            alpha=0.25,
            color=CLS_COLORS[c],
            label=CLS_NAMES[c],
            ax=axes[1, j],
        )
    axes[1, j].set_title(f"{PRETTY[k]}: после деления на p99 мозга своего кейса")
    axes[1, j].set_xlabel("нормированная интенсивность")
    axes[1, j].legend()
fig.tight_layout()
save_fig(fig, "eda_05_intensity_hist.png")


# ---------- Числа для выводов ----------
def cohen_d(a, b):
    sa, sb = a.std(), b.std()
    sp = np.sqrt((sa**2 + sb**2) / 2)
    return float((a.mean() - b.mean()) / sp) if sp > 0 else np.nan


eff = {}
for k in TARGET_MODALITIES:
    h = samples[(k, "healthy")]
    for c in ("ed", "ncr", "et"):
        if samples[(k, c)].size:
            eff[(PRETTY[k], CLS_NAMES[c])] = cohen_d(samples[(k, c)], h)
eff_tbl = pd.Series(eff).unstack().round(2)
print(
    "Размер эффекта (Cohen d) подобласти опухоли vs здоровая ткань (на пуле выборок):"
)
display(eff_tbl)
eff_tbl.to_csv(TABLE_DIR / "intensity_effect_size.csv")

best_flair = eff_tbl.loc["FLAIR"].abs().idxmax()
best_t1ce = eff_tbl.loc["T1ce"].abs().idxmax()
conclude(
    "гистограммы интенсивностей",
    [
        f"`FLAIR`: сильнее всего от здоровой ткани отличается класс «{best_flair}» (|d| = {eff_tbl.loc['FLAIR'].abs().max():.2f}); "
        f"`T1ce`: «{best_t1ce}» (|d| = {eff_tbl.loc['T1ce'].abs().max():.2f}). "
        "Это подтверждает роль модальностей: `FLAIR` — для границ `WT` (отёк), `T1ce` — для `ET`/`TC`.",
        "Распределения субрегионов заметно перекрываются между собой и со здоровой тканью — простой пороговой сегментации недостаточно, нужна контекстная 3D-модель.",
        "После нормализации по p99 мозга своего кейса формы распределений сближаются (сравните верхний и нижний ряд) — это аргумент в пользу поканальной нормализации каждого кейса.",
    ],
)

In [ ]:
# ---------- Межкейсовый разброс (доменный сдвиг между сканерами/протоколами) ----------
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
for j, k in enumerate(TARGET_MODALITIES):
    sns.histplot(ok[f"{k}_p50"], bins=40, ax=axes[j, 0], color="steelblue")
    axes[j, 0].set_title(f"{PRETTY[k]}: медиана интенсивности мозга по кейсам")
    sns.histplot(ok[f"{k}_p99"], bins=40, ax=axes[j, 1], color="indianred")
    axes[j, 1].set_title(f"{PRETTY[k]}: 99-й перцентиль мозга по кейсам")
    sns.scatterplot(
        x=ok[f"{k}_brain_mean"],
        y=ok[f"{k}_brain_std"],
        s=12,
        alpha=0.5,
        ax=axes[j, 2],
        color="seagreen",
    )
    axes[j, 2].set_title(f"{PRETTY[k]}: mean vs std мозга")
    axes[j, 2].set_xlabel("mean")
    axes[j, 2].set_ylabel("std")
fig.tight_layout()
save_fig(fig, "eda_06_inter_case_intensity_shift.png")


def cv(s):
    return float(s.std() / s.mean() * 100) if s.mean() else np.nan


shift_tbl = pd.DataFrame(
    {
        PRETTY[k]: {
            "медиана мозга, mean": ok[f"{k}_p50"].mean(),
            "медиана мозга, CV %": cv(ok[f"{k}_p50"]),
            "p99 мозга, mean": ok[f"{k}_p99"].mean(),
            "p99 мозга, CV %": cv(ok[f"{k}_p99"]),
            "p99 мозга, min": ok[f"{k}_p99"].min(),
            "p99 мозга, max": ok[f"{k}_p99"].max(),
        }
        for k in TARGET_MODALITIES
    }
).round(2)
display(shift_tbl)
shift_tbl.to_csv(TABLE_DIR / "intensity_inter_case_shift.csv")

max_ratio = {
    PRETTY[k]: ok[f"{k}_p99"].max() / max(ok[f"{k}_p99"].min(), 1e-6)
    for k in TARGET_MODALITIES
}
conclude(
    "межкейсовый сдвиг интенсивностей",
    [
        f"Коэффициент вариации p99-интенсивности мозга между кейсами: `FLAIR` {shift_tbl.loc['p99 мозга, CV %', 'FLAIR']:.1f}%, `T1ce` {shift_tbl.loc['p99 мозга, CV %', 'T1ce']:.1f}%; "
        f"отношение max/min p99: `FLAIR` {max_ratio['FLAIR']:.1f}×, `T1ce` {max_ratio['T1ce']:.1f}×.",
        "Абсолютная шкала интенсивностей не стандартизирована → без поканальной нормализации (`NormalizeIntensityd(nonzero=True, channel_wise=True)`) модель будет учить масштаб конкретного сканера. "
        "Это же главный источник риска доменного сдвига на данных другой клиники.",
    ],
)

# ---------- Контраст опухоль / здоровая ткань на уровне кейса ----------
contrast = pd.DataFrame({"case_id": ok["case_id"]})
for k in TARGET_MODALITIES:
    for c in ("ed", "ncr", "et"):
        contrast[f"{PRETTY[k]}|{c.upper()}"] = (
            ok[f"{k}_{c}_mean"].values / ok[f"{k}_healthy_mean"].values
        )
long = contrast.melt(
    id_vars="case_id", var_name="mod_class", value_name="ratio"
).dropna()
long[["mod", "cls"]] = long["mod_class"].str.split("|", expand=True)
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(
    data=long,
    x="cls",
    y="ratio",
    hue="mod",
    ax=ax,
    showfliers=False,
    palette=["#f59e0b", "#10b981"],
)
ax.axhline(1.0, color="k", ls="--", lw=1)
ax.set_xlabel("Подобласть опухоли")
ax.set_ylabel("mean(опухоль) / mean(здоровая ткань) на кейс")
ax.set_title(
    "Относительный контраст по кейсам: значения > 1 — опухоль ярче окружающей ткани"
)
save_fig(fig, "eda_07_tumor_vs_healthy_contrast.png")

med = long.groupby(["mod", "cls"])["ratio"].median().round(2)
display(med.unstack())
conclude(
    "контраст опухоль / здоровая ткань",
    [
        "Медианные отношения яркости (таблица выше) показывают, какая подобласть и в какой модальности видна лучше всего: чем дальше значение от 1, тем информативнее канал для этого субрегиона.",
        "Ожидаемая картина: `ET` существенно ярче здоровой ткани на `T1ce`, а `ED` — на `FLAIR`, т.е. каждая из двух модальностей отвечает за «свой» субрегион. "
        "Если для `NCR` контраст в обеих модальностях слабый или разнонаправленный, то именно ядро опухоли будет сложнее всего сегментировать без `T1`/`T2` (сокращённый протокол из 2 модальностей).",
    ],
)

## 6. Визуальный контроль: срезы `FLAIR` / `T1ce` с контурами масок

Выбираются репрезентативные кейсы: **медианный** по объёму `WT`, **самый крупный**, **самый мелкий** (ненулевой) и **кейс без `ET`** (если такой есть). Для каждого берётся аксиальный срез с максимальной площадью опухоли. Цвета контуров: `WT` — голубой, `TC` — жёлтый, `ET` — красный. Отдельно показываются 3 ортогональные проекции для одного кейса и заливка исходных меток.

In [ ]:
from matplotlib.colors import ListedColormap

CONTOUR_STYLE = {"WT": "#22d3ee", "TC": "#facc15", "ET": "#ef4444"}


def load_case(cid):
    f = index[cid]
    flair = np.asanyarray(nib.load(str(f["flair"])).dataobj).astype(np.float32)
    t1ce = np.asanyarray(nib.load(str(f["t1ce"])).dataobj).astype(np.float32)
    seg = np.rint(np.asanyarray(nib.load(str(f["seg"])).dataobj)).astype(np.int16)
    return np.nan_to_num(flair), np.nan_to_num(t1ce), seg


def get_slice(vol, plane, i):
    sl = [slice(None)] * 3
    sl[plane] = i
    return np.rot90(vol[tuple(sl)])


def draw_contours(ax, seg2d_regions, lw=1.2):
    for name, m2d in seg2d_regions.items():
        if m2d.any():
            ax.contour(
                m2d.astype(float),
                levels=[0.5],
                colors=[CONTOUR_STYLE[name]],
                linewidths=lw,
            )


def best_slice(wt, plane):
    other = tuple(a for a in range(3) if a != plane)
    area = wt.sum(axis=other)
    return int(np.argmax(area))


def show_img(ax, img, title):
    nzv = img[img > 0]
    vmax = np.percentile(nzv, 99.5) if nzv.size else 1.0
    ax.imshow(img, cmap="gray", vmin=0, vmax=vmax)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


# ---------- Выбор репрезентативных кейсов ----------
nz_ok = ok[ok["vox_wt"] > 0].sort_values("vox_wt")
picked = {
    f"медианный WT ({nz_ok['vox_wt'].median() / VOX_ML:.0f} мл)": nz_ok.iloc[
        len(nz_ok) // 2
    ]["case_id"],
    f"самый крупный WT ({nz_ok['vox_wt'].max() / VOX_ML:.0f} мл)": nz_ok.iloc[-1][
        "case_id"
    ],
    f"самый мелкий WT ({nz_ok['vox_wt'].min() / VOX_ML:.2f} мл)": nz_ok.iloc[0][
        "case_id"
    ],
}
no_et_cases = nz_ok[nz_ok["vox_et"] == 0]
if len(no_et_cases):
    picked["без ET"] = no_et_cases.iloc[0]["case_id"]
print("Выбранные кейсы:", picked)

rows = len(picked)
fig, axes = plt.subplots(rows, 3, figsize=(13, 4.2 * rows))
axes = np.atleast_2d(axes)
for r_i, (label, cid) in enumerate(picked.items()):
    flair, t1ce, seg = load_case(cid)
    wt, tc, et = labels_to_regions(seg)
    z = best_slice(wt, plane=2)
    regs = {
        "WT": np.rot90(wt[:, :, z]),
        "TC": np.rot90(tc[:, :, z]),
        "ET": np.rot90(et[:, :, z]),
    }
    show_img(axes[r_i, 0], np.rot90(flair[:, :, z]), f"{cid} | FLAIR, z={z}\n{label}")
    draw_contours(axes[r_i, 0], regs)
    show_img(axes[r_i, 1], np.rot90(t1ce[:, :, z]), f"{cid} | T1ce, z={z}")
    draw_contours(axes[r_i, 1], regs)
    cmap = ListedColormap(["black", "#a855f7", "#3b82f6", "#ef4444"])
    lab_img = np.rot90(
        np.select(
            [seg[:, :, z] == 1, seg[:, :, z] == 2, seg[:, :, z] == 4], [1, 2, 3], 0
        )
    )
    axes[r_i, 2].imshow(lab_img, cmap=cmap, vmin=0, vmax=3, interpolation="nearest")
    axes[r_i, 2].set_title("Метки: NCR (фиол.), ED (синий), ET (красный)", fontsize=9)
    axes[r_i, 2].axis("off")
handles = [Line2D([0], [0], color=c, lw=2, label=n) for n, c in CONTOUR_STYLE.items()]
fig.legend(handles=handles, loc="upper right", ncol=3, title="Контуры")
fig.tight_layout()
save_fig(fig, "eda_08_slices_with_contours.png")

# ---------- 3 ортогональные проекции для одного кейса ----------
cid = nz_ok.iloc[len(nz_ok) // 2]["case_id"]
flair, t1ce, seg = load_case(cid)
wt, tc, et = labels_to_regions(seg)
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
plane_names = {0: "Сагиттальная", 1: "Корональная", 2: "Аксиальная"}
for col, plane in enumerate((0, 1, 2)):
    i = best_slice(wt, plane)
    regs = {
        "WT": get_slice(wt, plane, i),
        "TC": get_slice(tc, plane, i),
        "ET": get_slice(et, plane, i),
    }
    for row, (name, vol_) in enumerate((("FLAIR", flair), ("T1ce", t1ce))):
        show_img(
            axes[row, col],
            get_slice(vol_, plane, i),
            f"{cid} | {name} | {plane_names[plane]}, срез {i}",
        )
        draw_contours(axes[row, col], regs)
fig.legend(handles=handles, loc="upper right", ncol=3)
fig.tight_layout()
save_fig(fig, "eda_09_orthogonal_views.png")

conclude(
    "визуальный контроль",
    [
        "Контуры `WT` совпадают с зоной гиперинтенсивности на `FLAIR` (отёк), а `ET` — с яркими кольцевыми структурами на `T1ce`; `TC` находится внутри `WT`, `ET` — внутри `TC`: вложенность `WT ⊃ TC ⊃ ET` соблюдается визуально.",
        "Разметка имеет сглаженные границы и местами захватывает области, где видимый контраст слабый, → при оценке нужна поправка на неопределённость границ (отсюда `HD95`, а не `HD`), а ошибки на границе опухоли нельзя считать полностью ошибками модели.",
        "На кейсе без `ET` на `T1ce` внутри опухоли нет ярких накапливающих контраст зон, что согласуется с низкозлокачественными глиомами; такие кейсы должны попадать в каждый фолд.",
        "У самого мелкого кейса опухоль — единицы вокселей/миллилитров: такие случаи легко «потерять» при `CropForegroundd`/случайных патчах без смещения на опухоль.",
    ],
)

## 7. Предварительный сплит на 5 фолдов

Каждый кейс BraTS — отдельный пациент (ID вида `BraTS2021_xxxxx`), поэтому группа = `case_id` исключает утечку между train/val. Для стратификации используется комбинация «есть ли `ET`» × «терциль объёма `WT`». Сплит фиксируется (`SEED`) и сохраняется в `docs/eda_tables/folds.csv` — это артефакт Этапа 1 дизайн-дока. Ниже проверяем сбалансированность фолдов.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

split = ok[["case_id", "vox_wt", "vox_tc", "vox_et"]].copy().reset_index(drop=True)
split["has_et"] = (split["vox_et"] > 0).astype(int)
split["wt_tercile"] = pd.qcut(
    split["vox_wt"].rank(method="first"), 3, labels=[0, 1, 2]
).astype(int)
split["strata"] = split["has_et"] * 3 + split["wt_tercile"]

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
split["fold"] = -1
for f_i, (_, val_idx) in enumerate(
    sgkf.split(split, y=split["strata"], groups=split["case_id"])
):
    split.loc[val_idx, "fold"] = f_i
assert (split["fold"] >= 0).all()
split[["case_id", "fold", "has_et", "wt_tercile"]].to_csv(
    TABLE_DIR / "folds.csv", index=False
)

fold_tbl = (
    split.groupby("fold")
    .agg(
        n_cases=("case_id", "count"),
        share_without_ET=("has_et", lambda s: 1 - s.mean()),
        wt_median_ml=("vox_wt", lambda s: s.median() / VOX_ML),
        et_median_ml=("vox_et", lambda s: s.median() / VOX_ML),
    )
    .round(3)
)
display(fold_tbl)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(x=fold_tbl.index, y=fold_tbl["n_cases"], ax=axes[0], color="steelblue")
axes[0].set_title("Кейсов в фолде")
sns.barplot(
    x=fold_tbl.index,
    y=fold_tbl["share_without_ET"] * 100,
    ax=axes[1],
    color="indianred",
)
axes[1].set_title("Доля кейсов без ET, %")
sns.barplot(x=fold_tbl.index, y=fold_tbl["wt_median_ml"], ax=axes[2], color="seagreen")
axes[2].set_title("Медианный объём WT, мл")
for a in axes:
    a.set_xlabel("фолд")
fig.tight_layout()
save_fig(fig, "eda_10_fold_balance.png")

spread_et = (
    fold_tbl["share_without_ET"].max() - fold_tbl["share_without_ET"].min()
) * 100
conclude(
    "сплит на фолды",
    [
        f"Размеры фолдов: {fold_tbl['n_cases'].min()}–{fold_tbl['n_cases'].max()} кейсов; разброс доли кейсов без `ET` между фолдами — {spread_et:.1f} п.п.; "
        f"медианный объём `WT` по фолдам: {fold_tbl['wt_median_ml'].min():.1f}–{fold_tbl['wt_median_ml'].max():.1f} мл.",
        "Фолды сбалансированы по наличию `ET` и размеру опухоли, поэтому метрики по фолдам будут сопоставимы, а `ET`-Dice не будет «пустым» ни на одном фолде.",
    ],
)

## 8. Сводка для дизайн-дока и сохранение графиков в `docs/images/`

Ячейка собирает ключевые числа в один блок — их нужно вручную перенести в места с тегом `[ожидает прогона на GPU]` в `docs/ml_system_design_doc.md`. Затем все графики упаковываются в архив для скачивания (и, при желании, копируются на Google Drive).

In [ ]:
def q(s, p):
    return float(s.quantile(p))


summary = {
    "n_cases_total": int(n_total),
    "n_cases_ok": int(n_ok),
    "n_cases_incomplete_in_archive": int(len(incomplete)),
    "disk_gb_flair_t1ce_seg": round(float(total_gb), 1),
    "shape_unique": sorted(ok["shape_flair"].map(fmt_tuple).unique().tolist()),
    "spacing_unique_flair": sorted(ok["zooms_flair"].map(fmt_tuple).unique().tolist()),
    "spacing_unique_t1ce": sorted(ok["zooms_t1ce"].map(fmt_tuple).unique().tolist()),
    "orientation_unique": sorted(ok["orientation"].unique().tolist()),
    "qc_violations": {k: int(v) for k, v in checks.items()},
    "brain_pct_of_volume_mean": round(float(bf.mean()), 1),
    "brain_bbox_pct_of_volume_mean": round(float(bb_frac.mean()), 1),
    "global_pct_of_brain_WT": round(float(global_frac["WT"]), 2),
    "global_pct_of_brain_TC": round(float(global_frac["TC"]), 2),
    "global_pct_of_brain_ET": round(float(global_frac["ET"]), 2),
    "wt_ml_median_p5_p95": [
        round(float(vol["WT"].median()), 1),
        round(q(vol["WT"], 0.05), 1),
        round(q(vol["WT"], 0.95), 1),
    ],
    "tc_ml_median_p5_p95": [
        round(float(vol["TC"].median()), 1),
        round(q(vol["TC"], 0.05), 1),
        round(q(vol["TC"], 0.95), 1),
    ],
    "et_ml_median_p5_p95": [
        round(float(vol["ET"].median()), 1),
        round(q(vol["ET"], 0.05), 1),
        round(q(vol["ET"], 0.95), 1),
    ],
    "n_without_ET": int(n_no_et),
    "pct_without_ET": round(n_no_et / n_ok * 100, 1),
    f"n_ET_lt_{ET_TINY_VOX}_vox": int(n_tiny_et),
    "n_without_TC": int(n_no_tc),
    "pct_multi_component_WT": round(multi, 1),
    "wt_bbox_max_mm_median_p95_max": [
        float(bbmax.median()),
        float(bbmax.quantile(0.95)),
        float(bbmax.max()),
    ],
    "intensity_p99_CV_pct": {
        PRETTY[k]: round(cv(ok[f"{k}_p99"]), 1) for k in TARGET_MODALITIES
    },
    "intensity_p99_max_over_min": {k: round(float(v), 1) for k, v in max_ratio.items()},
    "fold_sizes": fold_tbl["n_cases"].tolist(),
    "fold_share_without_ET_pct": (fold_tbl["share_without_ET"] * 100).round(1).tolist(),
}
with open(TABLE_DIR / "eda_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("=== ЧИСЛА ДЛЯ ДИЗАЙН-ДОКА ===")
print(json.dumps(summary, ensure_ascii=False, indent=2))

REPO_DIR = None
if REPO_DIR is not None:
    import shutil

    (REPO_DIR / "docs" / "images").mkdir(parents=True, exist_ok=True)
    (REPO_DIR / "docs" / "eda_tables").mkdir(parents=True, exist_ok=True)
    for p in IMG_DIR.glob("*.png"):
        shutil.copy2(p, REPO_DIR / "docs" / "images" / p.name)
    for p in TABLE_DIR.glob("*"):
        if p.name != "scan_cache.pkl":  # кеш тяжёлый и в репозиторий не нужен
            shutil.copy2(p, REPO_DIR / "docs" / "eda_tables" / p.name)
    print("Скопировано в", REPO_DIR)

# ---------- Архив для скачивания ----------
import shutil

bundle = shutil.make_archive(
    str(WORK_DIR / "eda_artifacts"),
    "zip",
    root_dir=WORK_DIR / "docs",
    base_dir="images",
)
tbl_bundle_dir = WORK_DIR / "docs" / "eda_tables_export"
tbl_bundle_dir.mkdir(exist_ok=True)
for p in TABLE_DIR.glob("*"):
    if p.name != "scan_cache.pkl":
        shutil.copy2(p, tbl_bundle_dir / p.name)
tbl_zip = shutil.make_archive(
    str(WORK_DIR / "eda_tables"),
    "zip",
    root_dir=WORK_DIR / "docs",
    base_dir="eda_tables_export",
)
print("Архивы:", bundle, tbl_zip)
print("Файлы графиков:", sorted(p.name for p in IMG_DIR.glob("*.png")))
